# Chavruta.AI — Embed Mishnah (Kaggle GPU)

Embeds **81K Mishnah + commentary chunks** with bge-m3 (dense + sparse).

Settings: Accelerator = **GPU T4** | Internet = **On**

Expected time: ~20-30 min on T4.

In [ ]:
!nvidia-smi

In [ ]:
!pip install -q -U FlagEmbedding huggingface_hub numpy

In [ ]:
import json
from pathlib import Path
from huggingface_hub import hf_hub_download

p = hf_hub_download(repo_id='Yehuda-Rubin/chavruta-torah-mixed',
                    filename='mishnah_chunks.json',
                    repo_type='dataset', local_dir='/kaggle/working')
print('downloaded:', p)

In [ ]:
raw = json.loads(Path('/kaggle/working/mishnah_chunks.json').read_text(encoding='utf-8'))
chunks = raw['chunks'] if isinstance(raw, dict) else raw
docs  = [c['document'] for c in chunks]
ids   = [c['id']       for c in chunks]
metas = [c['metadata'] for c in chunks]
print(f'chunks: {len(chunks):,}')
print('metadata:', raw.get('metadata', {}))

In [ ]:
from FlagEmbedding import BGEM3FlagModel
model = BGEM3FlagModel('BAAI/bge-m3', use_fp16=True, device='cuda')
print('model ready')

In [ ]:
import numpy as np, time, pickle

BATCH, MAX_LEN, CKPT_EVERY = 128, 512, 5000
ckpt = Path('/kaggle/working/embed_mishnah_ckpt.pkl')

if ckpt.exists():
    state = pickle.loads(ckpt.read_bytes())
    dense_parts, sparse_rows, start = state['dense'], state['sparse'], state['done']
    print(f'resuming from {start:,}')
else:
    dense_parts, sparse_rows, start = [], [], 0

t0 = time.time()
for s in range(start, len(docs), BATCH):
    batch = docs[s:s+BATCH]
    enc = model.encode(batch, batch_size=BATCH, max_length=MAX_LEN,
                       return_dense=True, return_sparse=True, return_colbert_vecs=False)
    dense_parts.append(np.asarray(enc['dense_vecs'], dtype='float32'))
    for w in enc['lexical_weights']:
        sparse_rows.append({int(t): float(v) for t, v in dict(w).items()})
    done = min(s + BATCH, len(docs))
    if done % CKPT_EVERY < BATCH or done == len(docs):
        ckpt.write_bytes(pickle.dumps({'dense': dense_parts, 'sparse': sparse_rows, 'done': done}))
    rate = done / max(time.time() - t0, 1e-9)
    eta = (len(docs) - done) / max(rate, 1e-9) / 60
    print(f'  {done:,}/{len(docs):,}  ({rate:.0f}/s, ETA {eta:.0f} min)', end='\r')

vecs = np.vstack(dense_parts)
norms = np.linalg.norm(vecs, axis=1, keepdims=True); norms[norms==0] = 1.0
vecs /= norms
print(f'\ndone: {vecs.shape} in {(time.time()-t0)/60:.1f} min')

In [ ]:
out = Path('/kaggle/working/out_mishnah'); out.mkdir(exist_ok=True)

np.save(out / 'corpus_vectors.npy', vecs)

with open(out / 'corpus_sparse.jsonl', 'w', encoding='utf-8') as f:
    for i, row in enumerate(sparse_rows):
        f.write(json.dumps({'i': i, 'sparse': row}) + '\n')

with open(out / 'corpus_meta.jsonl', 'w', encoding='utf-8') as f:
    for i, (cid, doc, meta) in enumerate(zip(ids, docs, metas)):
        f.write(json.dumps({'i': i, 'id': cid, 'document': doc, 'metadata': meta},
                           ensure_ascii=False) + '\n')

for p in sorted(out.iterdir()):
    print(f'{p.name:28s} {p.stat().st_size/1e6:8.1f} MB')

In [ ]:
import shutil
shutil.make_archive('/kaggle/working/mishnah_vectors', 'zip', out)
sz = Path('/kaggle/working/mishnah_vectors.zip').stat().st_size / 1e6
print(f'download: /kaggle/working/mishnah_vectors.zip  ({sz:.0f} MB)')

## After download
```powershell
Expand-Archive mishnah_vectors.zip -DestinationPath out_mishnah
python scripts/load_to_store.py --in out_mishnah --no-recreate --profile local
```